# Configuration

In [1]:
import os 
import pickle 

if True ^ os.getcwd().endswith('hte-and-targeting"'):
    os.chdir('..')


In [2]:
import random
import pandas as pd 
import numpy as np

from tqdm import tqdm

In [3]:
# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# visualization params
# label size
tick_label_size = 12
legend_label_size = 12
axis_label_size = 14
title_size = 18
# font
plt.rcParams['font.family'] = 'serif'

# model label map 
model_label_map = {
    'plugin': 'Plugin', 
    'standard_bootstrap': 'Standard Bootstrap', 
    'mn_bootstrap': 'm-out-of-n Bootstrap', 
    'num_bootstrap': 'Numerical Bootstrap'
}

In [4]:
import pymc as pm 

In [5]:
from core.variables import *
from core.dgp import SingleSegmentTreatmentSelection as SingleSegment
from core.dgp import MultipleSegmentsTreatmentSelection as MultipleSegments
from core.dgp import ContinuousSegments 
import core.treatment_selection_single_segment as tsss 
import core.treatment_selection_multiple_segments as tsmss

# Single Segment

In [30]:
sample_size = 100

ss_dgp = SingleSegment(**{
    'te_arr': np.array([1, 1.1]), 'noise_std': 1, 
    'response_type': 'continuous'
})
treatment_arr, outcome_arr = ss_dgp.sample(sample_size=100)

In [33]:
# Prior parameters for a1 and a2 (normal priors)
m1, s1 = outcome_arr[treatment_arr == 0].mean(), outcome_arr[treatment_arr == 0].std()
m2, s2 = outcome_arr[treatment_arr == 1].mean(), outcome_arr[treatment_arr == 1].std()

# -----------------------------
# 2. Define and Sample from the Model Using PyMC
# -----------------------------
with pm.Model() as model:
    # Priors for the treatment effects
    a1 = pm.Normal('a1', mu=m1, sigma=s1)
    a2 = pm.Normal('a2', mu=m2, sigma=s2)
    
    # Create the mean for each observation based on treatment assignment.
    # Note: We use a vectorized formulation.
    mu = pm.math.switch(pm.math.eq(treatment_arr, 1), a1, a2)
    
    # Likelihood for the observations
    y_obs = pm.Normal('y_obs', mu=mu, sigma=ss_dgp.noise_std, observed=outcome_arr)
    
    # Sample from the posterior
    trace = pm.sample(3000, tune=2000, target_accept=0.95, random_seed=42, return_inferencedata=False)

# -----------------------------
# 3. Compute Posterior Summaries from MCMC
# -----------------------------
posterior_mean_a1 = np.mean(trace['a1'])
posterior_mean_a2 = np.mean(trace['a2'])
print("\nPyMC3-based Bayesian Estimators:")
print("Posterior mean for a1 (Bayes estimator):", posterior_mean_a1)
print("Posterior mean for a2 (Bayes estimator):", posterior_mean_a2)

# Compute MAP estimates using PyMC3's find_MAP (note that for mixture models MAP can be less stable, but here it's straightforward)
map_estimate = pm.find_MAP(model=model)
print("MAP estimate for a1:", map_estimate['a1'])
print("MAP estimate for a2:", map_estimate['a2'])

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [a1, a2]


Output()

Sampling 4 chains for 2_000 tune and 3_000 draw iterations (8_000 + 12_000 draws total) took 12 seconds.



PyMC3-based Bayesian Estimators:
Posterior mean for a1 (Bayes estimator): 0.9996563975389808
Posterior mean for a2 (Bayes estimator): 0.6986381604716976


Output()

MAP estimate for a1: 0.9985226247550553
MAP estimate for a2: 0.6992500063983809


In [34]:
# empirical estimates
empirical_mean_a1 = np.mean(outcome_arr[treatment_arr == 0])
empirical_mean_a2 = np.mean(outcome_arr[treatment_arr == 1])
emp_te_arr = np.array([empirical_mean_a1, empirical_mean_a2])

# selection based on empirical estimates
best_treatment = 0 if empirical_mean_a1 > empirical_mean_a2 else 1

print(f"True value: {ss_dgp.te_arr[best_treatment]:.4f}")
print(f"Estimated value (no correction): {emp_te_arr[best_treatment]:.4f}")
print(f"Estimated value (Bayesian correction): {posterior_mean_a1 if best_treatment == 0 else posterior_mean_a2:.4f}")

True value: 1.1000
Estimated value (no correction): 1.0044
Estimated value (Bayesian correction): 0.6986


# Multiple Segments

In [ ]:
ms_dgp = MultipleSegments(**{
    'base_te_arr': np.array([1, 1.1]), 'n_segments': 5,
    'response_type': 'continuous', 'noise_std': 1,
})